# 01 제1유형

다음의 데이터는 IBM 직원들의 직무 정보와 퇴사 여부에 대한 데이터이다.

**데이터 URL**
```
https://raw.githubusercontent.com/YoungjinBD/dataset/main/HR-Employee-Attrition.csv
```

| 컬럼 | 설명 |
|---|---|
| Attrition | 퇴사 여부 (`Yes`: 퇴사, `No`: 퇴사하지 않음) |
| 기타 | 직무/인구통계/근무 관련 변수 |

In [ ]:
import pandas as pd
import numpy as np

# 데이터 로드
data_url = "https://raw.githubusercontent.com/YoungjinBD/dataset/main/HR-Employee-Attrition.csv"
hr = pd.read_csv(data_url)
hr.head()

## 문제 ① Attrition 수치화 및 범주별 건수

해당 데이터에서 `Attrition`은 종속변수이다.  
Attrition의 값을 **수치형**으로 변환해 새로운 컬럼으로 추가하고,  
범주별 **레코드 수**를 구하시오.

- `Yes` → `1` (퇴사)
- `No` → `0` (퇴사하지 않음)

> 출력: Attrition(0/1) 범주별 건수

In [1]:
hr['Attrition_num'] = hr['Attrition'].map({'Yes': 1, 'No': 0})
counts = hr['Attrition_num'].value_counts()
counts

Attrition_num
0    316
1     57
Name: count, dtype: int64

## 문제 ② 자료형별 컬럼 수 및 단일값 범주형 제거

데이터셋에서 **자료형(dtype)별 컬럼 수**를 계산하고,  
범주형 변수 중 **유일한 값이 1개뿐인 변수**를 찾아 데이터에서 제거하시오.

> 출력: dtype별 컬럼 수, 제거 대상 컬럼명, 제거 후 데이터 shape

In [2]:
dtype_counts = hr.dtypes.value_counts()

cat_cols = hr.select_dtypes(include='object').columns
single_val_cols = [c for c in cat_cols if hr[c].nunique() == 1]

hr = hr.drop(columns=single_val_cols)

print(dtype_counts)
print("제거 대상 컬럼:", single_val_cols)
print("제거 후 shape:", hr.shape)

int64     27
object     9
Name: count, dtype: int64
제거 대상 컬럼: ['Over18']
제거 후 shape: (373, 35)


## 문제 ③ 수치형 상관분석 및 고상관 변수 제거

원본 데이터에서 **수치형 변수만** 추출한 데이터프레임을 만들고,  
각 변수 간 **피어슨 상관계수**를 구하시오.  
상관계수가 **0.9 이상**인 두 변수를 찾아 그 중 **한 변수를 제거**하시오.

> 출력: 상관계수 0.9 이상인 변수 쌍, 제거한 컬럼명, 제거 후 shape

In [3]:
num_df = hr.select_dtypes(include=np.number)
corr = num_df.corr(method='pearson')

pairs = []
cols = corr.columns
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        v = corr.iloc[i, j]
        if abs(v) >= 0.9:
            pairs.append((cols[i], cols[j], round(v, 4)))

to_drop = list(dict.fromkeys(p[1] for p in pairs))
num_df = num_df.drop(columns=to_drop)

print("고상관 변수 쌍(|r|>=0.9):", pairs)
print("제거한 컬럼:", to_drop)
print("제거 후 shape:", num_df.shape)

고상관 변수 쌍(|r|>=0.9): [('JobLevel', 'MonthlyIncome', 0.9506)]
제거한 컬럼: ['MonthlyIncome']
제거 후 shape: (373, 26)
